# Reproducing Huszar et al. (2022) from dandiset 000552

**Paper.** Huszar R, Zhang Y, Blockus H, Buzsaki G (2022). Preconfigured dynamics in the
hippocampus are guided by embryonic birthdate and rate of neurogenesis. *Nature Neuroscience* 25,
1201-1212. [doi:10.1038/s41593-022-01138-x](https://doi.org/10.1038/s41593-022-01138-x). This
attempt used the full text of the author manuscript in PubMed Central (PMC10807234); the version of
record is not open access. The Supplementary Tables were not available to it.

**Dandiset.** [DANDI:000552](https://dandiarchive.org/dandiset/000552/0.230630.2304), version
0.230630.2304. The paper cites no DANDI version (its data statement points to the Buzsaki lab
repository). This is the only published version and the most recent one, so it is pinned. The
draft differs from it only by three small BIDS files (`README`, `dataset_description.json`,
`bids-validator-config.json`) that the analysis does not use; the data check below shows this.

**What the dandiset holds.** 104 processed files (one per session, 17 mice) with sleep states and
sharp-wave ripples, of which 65 (11 mice) also hold spike-sorted units, position and maze trials;
and 13 raw files. The units carry no label of light-responsiveness (birthdate), no cell type and no
waveforms, and no optogenetic pulse times are stored. The paper's central comparisons, same- against
different-birthdate neurons and light-responsive neurons across birthdates, therefore cannot be
made. What can be tested is the description of the dataset and the findings that use all pyramidal
neurons on the familiar and novel mazes.

**Code read.** The paper's analysis code, https://github.com/rhuszar/optotagCA1_final at commit
51320d45c5ee41147048128df333f251379bf755 (MATLAB, with some result files). It sets the figure-eight
segments and the stem exclusion used here (see `huszar.py`). No data are taken from it.

No random numbers are drawn in the claim sections; the only random draw is the time shuffle of the
peri-ripple control in the data check, seeded in `huszar.py`.

| Claim | Status |
|---|---|
| cohort-birthdates | tested |
| sessions-durations | tested |
| spwr-duration | tested |
| unit-count | tested |
| maze-cohort | tested |
| maze-task | tested |
| novel-explorations | tested |
| fig5e-pv-decorrelation | tested |
| fig6c-theta-rate-preserved | tested |
| fig6c-novel-ratemap-corr-higher | tested |
| fig6d-info-lower-novel | tested |
| fig6d-rate-lower-novel | tested |
| ed2-light-responsive-count, fig1e, fig1f, fig1g, fig1h, fig2b, fig2c, fig2d, fig2f, fig2g, fig3c, fig3e, fig5b, fig5c, fig5f, fig6e, fig6f | blocked: no light-responsiveness labels or pulse times (data missing) |
| fig5d, fig7b, fig7ce | blocked: also need unit positions or cell types (data missing) |
| fig4d-model-fit | blocked: depends on fig3e |
| fig1c-radial-depth | blocked: histology not in the dandiset (data missing) |
| fig7fg-convergence-cofiring | blocked: connection detection does not match the paper's (methods underspecified) |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dandi.dandiapi import DandiAPIClient
from scipy import stats

import huszar as h

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

def familiar(s):
    # the familiar figure-eight recording of a session: its first maze recording
    return next(r for r in h.recordings(s) if r["familiar"])


started = time.time()
assets = h.list_assets()
sessions = h.load_all()
print(f"loaded {len(sessions)} processed files of dandiset {h.DANDISET_ID} version {h.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
proc, raw = assets[assets.kind == "processed"], assets[assets.kind == "raw"]
print(f"{len(assets)} files: {len(proc)} processed (behavior+ecephys), {len(raw)} raw_ecephys; "
      f"{assets.size_gb.sum():.0f} GB in all, {proc.size_gb.sum():.1f} GB processed")

info = pd.DataFrame([{
    "session": s["session_id"], "subject": s["subject"], "birthdate": s["birthdate"], "sex": s["sex"],
    "age": s["age"], "date": s["start"][:10], "duration_min": round(s["end_s"] / 60, 1),
    "states": len(s["states"]), "ripples": len(s["ripples"]),
    "units": len(s["n_spikes"]) if s["has_units"] else 0, "position": s["has_position"],
    "recordings": ",".join(map(str, sorted(s["trials"].recordings.unique()))) if s["has_position"] else "",
    "trials": len(s["trials"]) if s["has_position"] else 0} for s in sessions])
subjects = info.groupby("subject").agg(
    birthdate=("birthdate", "first"), sex=("sex", lambda x: x.value_counts().idxmax()),
    sex_by_file=("sex", lambda x: ", ".join(f"{k} {v}" for k, v in x.value_counts().items())), sessions=("session", "size"),
    with_units=("units", lambda u: int((u > 0).sum())), units=("units", "sum"),
    with_position=("position", "sum")).reset_index()
subjects["raw_files"] = subjects.subject.map(raw.subject.str.replace("-", "_").value_counts()).fillna(0).astype(int)
print(subjects.to_string(index=False))
print("\nMice whose sex field differs between their files:", subjects.loc[subjects.sex_by_file.str.contains(","), ["subject", "sex_by_file"]].values.tolist())
print("Sex of each mouse is taken as the value in most of its files (column sex).")
print("\nAge field values:", info.age.unique().tolist())
print("Archive summary: numberOfSubjects 17, species Mus musculus, 117 files.")

The archive's summary (17 subjects, 117 files) matches the files. The birthdate (embryonic day of
electroporation) is stored nowhere in the files except as the prefix of the subject id (`e13` for
E13.5); the age is a range (P3M/P6M) for every mouse. The sex field is not the same in all files of a mouse for
e13_1f1 and e13_1m1 (the counts by file are printed above); the sex of a mouse is taken here as the
value in most of its files. Six mice (e13_1f1 and e13_1m1 at E13.5, e14_1m1 at E14.5, and e15_4m1, e15_4m2 and e15_5m1 at
E15.5) have 39 files between them that hold sleep states and ripples but no units and no position.

In [ ]:
draft = {a.path: a.size for a in DandiAPIClient().get_dandiset(h.DANDISET_ID, "draft").get_assets()}
pub = dict(zip(assets.path, assets["size"]))
print("Files only in the draft:", sorted(set(draft) - set(pub)))
print("Files only in the published version:", sorted(set(pub) - set(draft)))
print("Files in both with different sizes:", [p for p in pub if p in draft and draft[p] != pub[p]])

The draft adds three small BIDS files and changes no data file, so the published version holds
everything the draft holds that the analysis could use.

In [ ]:
print(info.to_string(index=False))

In [ ]:
units_files = [s for s in sessions if s["has_units"]]
print("Columns of the units tables:", sorted({tuple(s["units_columns"]) for s in units_files}))
for g in ("acquisition", "stimulus/presentation", "analysis", "intervals"):
    print(f"Contents of {g} across files:", sorted({tuple(s["groups"].get(g, [])) for s in sessions}))
mods = pd.Series([str(s["processing"]) for s in sessions]).value_counts()
print("\nProcessing modules and their contents (number of files):")
for k, v in mods.items():
    print(f"  {v:3d}  {k}")
import re
print("\nForms of unit_name over all units (digits as #):", sorted({re.sub(r"\d+", "#", n) for s in units_files for n in s["unit_names"]}))
print("Files where unit_name is the clu_id as text:", sum(all(n == str(c) for n, c in zip(s["unit_names"], s["clu_ids"])) for s in units_files), "of", len(units_files))
for s in units_files[:3]:
    print(f"  {s['session_id']}: unit_name {s['unit_names'][:5]}, clu_id {s['clu_ids'][:5].tolist()}, group_id {s['unit_groups'][:5]}")
same = sum(set(s["unit_groups"]) <= set(s["electrode_groups"]) for s in units_files)
print(f"\nFiles in which every unit group_id names an electrode group: {same} of {len(units_files)}")
print("Example (first file with units):", sorted(set(units_files[0]["unit_groups"])), "against",
      units_files[0]["electrode_groups"])

Nothing in the files marks a unit as light-responsive or gives its cell type; the units tables
hold only spike times, a unit name, a cluster id and a group id, and the forms of the unit names and
cluster ids printed above carry no cell type or light-response code. No optogenetic stimulation times
are stored (stimulus and acquisition are empty in every processed file; the raw files are checked
below). The units have no waveforms and no link to the electrodes table, and in some files their
`group_id` values ("Group3") are not the names of the electrode groups ("Group 3"), so a unit's
place on the probe cannot be recovered either.

In [ ]:
rates = np.concatenate([h.mean_rates(s) for s in units_files])
print(f"Units: {len(rates)} in {len(units_files)} files")
print("Session mean firing rate (Hz), percentiles 0, 5, 25, 50, 75, 95, 100:",
      np.percentile(rates, [0, 5, 25, 50, 75, 95, 100]).round(2).tolist())
print(f"Fraction at or below {h.PYR_MAX_RATE_HZ} Hz (this attempt's putative pyramidal rule): {np.mean(rates <= h.PYR_MAX_RATE_HZ):.3f}"
      f"; at or below 10 Hz: {np.mean(rates <= 10):.3f}; paper's pyramidal fraction 7411/9471 = {7411 / 9471:.3f}")
print("Spike times sorted within every unit in every file:", all(s["spikes_sorted"] for s in units_files))
late = [(s["session_id"], round(s["last_spike"] - s["end_s"], 1)) for s in units_files if s["last_spike"] > s["end_s"] + 1]
print("Files whose last spike is more than 1 s after the last sleep-state or ripple interval:", late)
print("Earliest first spike over files (s):", round(min(s["first_spike"] for s in units_files), 3))
dur = np.concatenate([s["ripples"].duration.values for s in sessions])
print(f"\nRipples: {len(dur)} in {len(sessions)} files; durations {dur.min() * 1000:.1f} to {dur.max() * 1000:.1f} ms")
dd = np.concatenate([(s["ripples"].stop - s["ripples"].start - s["ripples"].duration).values for s in sessions])
print(f"stop - start - ripple_durations: largest absolute difference {np.abs(dd).max() * 1000:.3f} ms")
inside = np.concatenate([((s["ripples"].peak >= s["ripples"].start) & (s["ripples"].peak <= s["ripples"].stop)).values for s in sessions])
print(f"Ripple peaks inside their start-stop interval: {inside.mean():.4f}")
labels = pd.Series(np.concatenate([s["states"].label.values for s in sessions])).value_counts()
print("Sleep-state labels (intervals over all files):", labels.to_dict())
print("Files with at least one Non-REM interval:", sum((s["states"].label == "Non-REM").any() for s in sessions), "of", len(sessions))

Spike times are sorted. In 57 files the last spike comes 1.2 to 5.2 s after the end of the last
sleep-state or ripple interval, so the session length used here (that end) is short by a few
seconds. Ripple durations lie within the 20 to 200 ms limits of the Methods, and the stored
durations equal stop minus start. Two files (e13_1f1_200305 and e13_1f1_200307) hold a single
Awake interval for the whole session, so sleep was not scored in them. Units are in seconds and
are consistent with the session lengths. The fraction of units at or below 5 Hz (the rule used here
for putative pyramidal cells, as the files hold no cell type) is compared with the paper's fraction
of pyramidal cells.

In [ ]:
rows = []
for s in sessions:
    if not s["has_position"]:
        continue
    for r in h.recordings(s):
        p = r["pos"]
        rows.append({"session": s["session_id"], "recording": r["recording"], "familiar": r["familiar"], "kind": r["kind"],
                     "trials": len(r["trials"]), "pos_unit": s["pos_unit"],
                     "dt_ms": round(1000 * np.median(np.diff(p.t)), 2),
                     "lin_max_in_trials": round(r["track_max_cm"], 1),
                     "x_range": f"{np.nanmin(p.x):.2f}-{np.nanmax(p.x):.2f}", "y_range": f"{np.nanmin(p.y):.2f}-{np.nanmax(p.y):.2f}",
                     "speed_p95": round(np.nanpercentile(p.speed, 95), 1)})
recs = pd.DataFrame(rows)
print(recs.to_string(index=False))
nan_trials = sum(int((s["trials"].start_time.isna() | s["trials"].stop_time.isna()).sum()) for s in sessions if s["has_position"])
print(f"\nTrials with no start or stop time: {nan_trials} of {sum(len(s['trials']) for s in sessions if s['has_position'])}")
fam_flag = []
for s in sessions:
    if not s["has_position"]:
        continue
    first = min(s["trials"].recordings.unique(), key=lambda r: s["trials"][s["trials"].recordings == r].start_time.min())
    for _, t in s["trials"].iterrows():
        fam_flag.append({"first_recording": t.recordings == first, "is_familiar_maze": t.is_familiar_maze,
                         "single_recording_session": s["trials"].recordings.nunique() == 1})
ff = pd.DataFrame(fam_flag)
print("\nis_familiar_maze against whether the trial is in the session's first maze recording:")
print(pd.crosstab([ff.single_recording_session, ff.first_recording], ff.is_familiar_maze).to_string())

Position is stated in cm. The familiar figure-eight recordings span about 90 by 90 cm (the Methods
give a 79 by 79 cm maze, so the tracked head positions extend somewhat beyond it or the scale is
approximate), and the linearized position inside trials reaches about 205 cm, the trial length of
the Methods. In several novel recordings the x and y values lie between 0 and 1, so they are not in
cm as stated; their linearized position (0 to 100, 110 or 164 cm) looks like cm. Position is
sampled every 33.3 ms (30 Hz) in most recordings and every 40 ms (25 Hz) in some novel ones.
`is_familiar_maze` is True for only the first two trials of sessions with a single maze
recording, where every trial is on the familiar maze, so the column is wrong there; this attempt
takes the first maze recording of a session as the familiar figure-eight (Methods: the familiar
maze was always run first).

In [ ]:
raw_info = pd.DataFrame([h.raw_file_info(a) for a in raw.asset])
raw_info["session_date_in_name"] = raw_info.path.str.extract(r"-(\d{6})-raw")[0]
print(raw_info.to_string(index=False))
big = [s for s in sessions if "lfp" in s]
for s in big:
    print(f"\nProcessed file {s['session_id']} also holds an LFP series: {s['lfp']}")
    rawa = raw[raw.path.str.contains(s["session_id"].replace("_", "-") + "-raw")].asset.iloc[0]
    proca = proc[proc.path.str.contains(s["session_id"].replace("_", "-") + "_behavior")].asset.iloc[0]
    a, b = h.lfp_samples(proca, 1_000_000, 2000, 5), h.lfp_samples(rawa, 1_000_000, 2000, 5)
    print("Same 2000 samples (channel 5) as the raw file's 1250 Hz LFP:", bool(np.array_equal(a, b)))

In [ ]:
# Everything one raw file holds, and whether the end of its session holds clipped (flat or zeroed) stretches
ra = raw.iloc[-1]
f = h.open_asset(ra.asset)
items = []
def walk(g, prefix=""):
    for k in g.keys():
        o = g[k]
        if hasattr(o, "keys"):
            if prefix + k != "specifications":
                walk(o, prefix + k + "/")
        else:
            items.append(f"  {prefix}{k} {o.shape} {o.dtype}")
walk(f)
print(ra.path); print("\n".join(items))
d = f["acquisition/ElectricalSeries/data"]
rate = float(f["acquisition/ElectricalSeries/starting_time"].attrs["rate"])
ses = next(s for s in sessions if s["session_id"].replace("_", "-") in ra.path)
print(f"Raw length {d.shape[0] / rate:.1f} s; processed session {ses['session_id']} ends at {ses['end_s']:.1f} s")
scan = []
for k in range(1, 11):
    st = d.shape[0] - int(k * 180 * rate)
    x = d[st:st + int(2 * rate), :8].astype(int)
    flat = (np.diff(x, axis=0) == 0).all(axis=1)
    ends = np.diff(np.r_[0, flat.astype(int), 0])
    runs = np.where(ends == -1)[0] - np.where(ends == 1)[0]
    scan.append({"start_s": round(st / rate), "flat_steps": int(flat.sum()), "longest_flat": int(runs.max()) if len(runs) else 0,
                 "zero_on_all_8": int((x == 0).all(axis=1).sum())})
f.close()
scan = pd.DataFrame(scan)
print(f"Last 30 min, a 2 s window every 3 min, channels 0-7: samples unchanged on all 8 channels {scan.flat_steps.sum()}, "
      f"longest such run {scan.longest_flat.max()}, samples zero on all 8 {scan.zero_on_all_8.sum()}")


The raw files hold the wideband signal (30 kHz) for 11 sessions and an LFP (1250 Hz) for 12. The
listing of one raw file above shows no stimulation, TTL or event series, only the two voltage series
and the electrodes table. The Methods say light pulses were given at the end of each session and the
raw data were clipped around each pulse. In that file the wideband signal ends 2 s after the
processed session, and ten 2 s windows over its last 30 min hold no flat or zeroed stretch on the
eight channels read, where clipping around pulses given every 200 ms would leave several in each
window. This is consistent with the stimulation period not being in the deposited recording; only
this one file was scanned. Twelve of the 13 raw files give a session start of 2011-08-18,
which cannot be right for sessions recorded in 2021 and 2022 (the date in the file name and in the
processed file). The processed
file of e16_3m1_210201 holds a copy of the raw file's 1250 Hz LFP, but labelled 30,000 Hz and with a
conversion factor of 3.05e-13 V; both attributes are wrong. No analysis here uses the LFP.

In [ ]:
# 1. Ripples and spikes: population rate around ripple peaks, against the same peaks shifted at random by up to 60 s.
centre = (h.RIPPLE_PSTH_EDGES[:-1] > -0.02) & (h.RIPPLE_PSTH_EDGES[:-1] < 0.02)
base = np.abs(h.RIPPLE_PSTH_EDGES[:-1]) >= 0.4
ratio = np.array([s["ripple_psth"][centre].mean() / s["ripple_psth"][base].mean() for s in units_files])
ratio_sh = np.array([s["ripple_psth_shuffled"][centre].mean() / s["ripple_psth_shuffled"][base].mean() for s in units_files])
print(f"Population rate within 20 ms of ripple peaks over rate 0.4-0.5 s away: median {np.median(ratio):.2f} "
      f"(range {ratio.min():.2f} to {ratio.max():.2f}); shifted peaks: median {np.median(ratio_sh):.2f} "
      f"(range {ratio_sh.min():.2f} to {ratio_sh.max():.2f})")

# 2. Ripples and sleep states: ripple rate in each state.
rr = []
for s in sessions:
    st = s["states"]
    for lab in st.label.unique():
        sel = st[st.label == lab]
        n = sum(((s["ripples"].peak >= a) & (s["ripples"].peak < b)).sum() for a, b in zip(sel.start, sel.stop))
        rr.append({"session": s["session_id"], "state": lab, "per_min": n / ((sel.stop - sel.start).sum() / 60)})
rr = pd.DataFrame(rr).pivot(index="session", columns="state", values="per_min")
print("\nRipples per minute by sleep state, median over files:", rr.median().round(2).to_dict())
print("Files where the REM rate is the lowest of the three:", int((rr["REM"] < rr[["Awake", "Non-REM"]].min(axis=1)).sum()), "of", len(rr))

# 3. Trials, position and sleep states.
starts, ends, awake = [], [], []
for s in sessions:
    if not s["has_position"]:
        continue
    p = s["pos"]
    for k, g in p[p.trial >= 0].groupby("trial"):
        starts.append(g.lin.iloc[0]); ends.append(g.lin.iloc[-1])
    tr = s["trials"].dropna(subset=["start_time", "stop_time"])
    aw = s["states"][s["states"].label == "Awake"]
    tot = (tr.stop_time - tr.start_time).sum()
    cov = sum(max(0, min(b, d) - max(a, c)) for a, b in zip(tr.start_time, tr.stop_time) for c, d in zip(aw.start, aw.stop))
    awake.append(cov / tot)
print(f"\nLinearized position at the first sample of a trial: median {np.median(starts):.1f} cm; at the last: {np.median(ends):.1f} cm")
print(f"Fraction of trial time labelled Awake: median {np.median(awake):.3f}, lowest {np.min(awake):.3f}")

In [ ]:
# 4. Spikes and position: spatial information of all units on the first maze recording, with spikes shifted in time.
shifts = [-2.0, -0.5, 0.0, 0.5, 2.0]
shift_rows = []
for s in units_files:
    if not s["has_position"]:
        continue
    rec = familiar(s)
    row = {"session": s["session_id"]}
    for sh in shifts:
        m = h.rate_maps(s["maze_spikes"], rec, shift_s=sh)
        row[sh] = np.nanmedian(h.skaggs_information(m["maps"].reshape(len(m["maps"]), -1), m["occupancy"].ravel()))
    shift_rows.append(row)
shift_df = pd.DataFrame(shift_rows).set_index("session")
best = shift_df.idxmax(axis=1)
print("Median spatial information (bits/s) by time shift of spikes against position, median over sessions:",
      shift_df.median().round(3).to_dict())
print("Sessions whose information peaks at zero shift:", int((best == 0.0).sum()), "of", len(best),
      "; others:", best[best != 0.0].to_dict())

The checks are consistent with a shared clock. Population firing rises about five-fold at ripple
peaks and not at randomly shifted peaks, so the ripple tables share the spikes' clock. Ripples are
rarest in REM sleep in 101 of 104 files, as expected if the state intervals share that clock. Trials
begin near 0 cm and end near 205 cm of linearized position and fall in Awake intervals, so the
trials share the position clock. Spatial information is highest with no shift between spikes and
position in 53 of 65 sessions and at a shift of 0.5 s in the other 12, well above its value at 2 s,
so spikes and position share a clock to within about half a second; a finer offset is not ruled
out. The 39 files without units could only be checked for ripples against sleep states.

In [ ]:
# Recordings deposited more than once: identical spike counts per unit, or identical ripple tables.
sig_units = {}
for s in units_files:
    sig_units.setdefault(tuple(s["n_spikes"]), []).append(s["session_id"])
sig_rip = {}
for s in sessions:
    sig_rip.setdefault((len(s["ripples"]), round(float(s["ripples"].peak.sum()), 3)), []).append(s["session_id"])
print("Files sharing identical per-unit spike counts:", [v for v in sig_units.values() if len(v) > 1])
print("Files sharing identical ripple tables:", [v for v in sig_rip.values() if len(v) > 1])
two = info[info.session.str.startswith("e15_5m1_200313")]
print("\nThe two files of e15_5m1 on 2020-03-13:")
print(two.to_string(index=False))

No recording is deposited twice. The two files of mouse e15_5m1 dated 2020-03-13 differ in length and
in their ripples, so they are two recordings of one day (the paper's code lists both).

## Claims about the dataset

### cohort-birthdates

> Chronic recordings were made from 17 adult mice (12 males, 5 females) electroporated at E13.5
> (n = 4), E14.5 (n = 3), E15.5 (n = 6) and E16.5 (n = 4).

In [ ]:
print("Mice:", subjects.subject.nunique())
print("By sex (each mouse's value in most of its files):", subjects.sex.value_counts().to_dict())
first_file = info.groupby("subject").sex.first()
print("By sex, read from each mouse's first file:", first_file.value_counts().to_dict())
print("Sex by file for the mice whose files differ:", subjects.loc[subjects.sex_by_file.str.contains(","), ["subject", "sex_by_file"]].values.tolist())
print("By birthdate:", subjects.birthdate.value_counts().sort_index().to_dict())
print("Paper: 17 mice; 12 M, 5 F; E13.5 4, E14.5 3, E15.5 6, E16.5 4.")
print(pd.crosstab(subjects.birthdate, subjects.sex).to_string())
subjects["sex_from_id"] = subjects.subject.str.extract(r"_\d+([fm])\d+$")[0].str.upper()
print("\nSex implied by the subject id (the letter before the last digit, f or m):", subjects.sex_from_id.value_counts().to_dict())
print("Mice whose sex field disagrees with their id:", subjects.loc[subjects.sex != subjects.sex_from_id, ["subject", "sex", "sex_from_id"]].values.tolist())

**Result: reproduced.** The dandiset holds 17 mice with the stated birthdates (E13.5 4, E14.5 3,
E15.5 6, E16.5 4). Taking each mouse's sex from most of its files gives 12 males and 5 females, as
the paper states. The sex field differs between files of two mice (e13_1f1 is Male in 2 files and
Female in 3; e13_1m1 is Male in 9 and Female in 3), so reading only the first file of each mouse
gives 13 and 4. The ids, which carry f or m, also give 12 and 5.

### sessions-durations

> The recordings comprise 92 sessions lasting 113.7 to 475.6 min (median 309.1 min), each including
> a homecage period of sleep and wake.

In [ ]:
d = info.duration_min
print(f"Processed files (sessions): {len(info)}; paper: 92 (Methods) or 96 (Figure 1e and Figure 7f).")
print(f"Duration: shortest {d.min():.1f} min ({info.session[d.idxmin()]}), longest {d.max():.1f} min ({info.session[d.idxmax()]}), median {d.median():.1f} min")
print(f"Durations within 102.3-523.2 min: {int(d.between(102.3, 523.2).sum())} of {len(d)}; within 113.7-475.6 min: {int(d.between(113.7, 475.6).sum())}")
print("Sessions longer than 475.6 min:", info.loc[d > 475.6, ["session", "duration_min"]].values.tolist())
print("Sessions with a duration equal (to 0.1 min) to the paper's bounds:", info.loc[d.round(1).isin([113.7, 475.6]), ["session", "duration_min"]].values.tolist())
excl = info[~info.subject.isin(["e15_13f1", "e16_3m2"])]
print(f"\nVariant: without e15_13f1 and e16_3m2, the two mice absent from the paper code's homecage results: "
      f"{len(excl)} sessions, median {excl.duration_min.median():.1f} min, range {excl.duration_min.min():.1f}-{excl.duration_min.max():.1f}")
print("Sessions per birthdate (all files):", info.birthdate.value_counts().sort_index().to_dict(),
      "; Figure 1e: E13.5 30, E14.5 24, E15.5 24, E16.5 18")
print("Sessions per birthdate (without e15_13f1 and e16_3m2):", excl.birthdate.value_counts().sort_index().to_dict())
no_sleep = [(s["session_id"], s["states"].label.value_counts().to_dict()) for s in sessions if "Non-REM" not in set(s["states"].label)]
print(f"Files without a Non-REM interval: {len(no_sleep)} of {len(sessions)}: {no_sleep}")

**Result: partially reproduced.** The 104 files are within 10% of the paper's other session
count (96, from Figure 1e and Figure 7f), though not of the 92 in the Methods; their median length is
319.6 min (paper 309.1) and all lie within the widened range, from 113.7 to 494.2 min. Without
e15_13f1 and e16_3m2 there are 96 sessions whose per-birthdate counts equal those of Figure 1e (30,
24, 24, 18) and whose lengths run from 113.7 to 475.6 min, the paper's bounds, with a median of 309.9
min. Two files of e13_1f1 hold no scored sleep (one Awake interval for the whole session), so not
every session holds the sleep the claim states; that part fails.

### spwr-duration

> SPW-Rs were detected with duration limits of 20 to 200 ms, and their median duration across all
> recordings was 36.4 ms.

In [ ]:
per = pd.DataFrame([{"session": s["session_id"], "subject": s["subject"], "n": len(s["ripples"]),
                     "median_ms": 1000 * s["ripples"].duration.median()} for s in sessions])
print(f"(a) median over all {len(dur)} ripples pooled: {1000 * np.median(dur):.1f} ms")
print(f"(b) median of the {len(per)} per-session medians: {per.median_ms.median():.1f} ms")
print(f"Ripples within 20-200 ms: {np.mean((dur >= 0.02) & (dur <= 0.2)):.4f}; shortest {1000 * dur.min():.1f}, longest {1000 * dur.max():.1f} ms")
print("Paper: 36.4 ms (tolerance 32.8-40.0).")
print("\nBy mouse (median of session medians, ms, and ripples):")
print(per.groupby("subject").agg(sessions=("n", "size"), ripples=("n", "sum"), median_ms=("median_ms", "median")).round(1).to_string())

**Result: reproduced.** The median ripple duration is 36.0 ms under both readings (paper 36.4 ms),
and all 494,578 ripples last 20.0 to 199.2 ms.

### unit-count

> The dataset includes 9,471 well-isolated units, 7,411 putative pyramidal cells and 2,060 putative
> interneurons.

In [ ]:
print(f"Units in the dandiset: {len(rates)} in {len(units_files)} of {len(sessions)} files; paper 9,471 (tolerance 8,524-10,418)")
print(f"Fraction at or below {h.PYR_MAX_RATE_HZ} Hz: {np.mean(rates <= h.PYR_MAX_RATE_HZ):.3f} (paper's pyramidal fraction 0.783)")
print("\nBy mouse:")
print(subjects[["subject", "birthdate", "sessions", "with_units", "units"]].to_string(index=False))
nou = subjects[subjects.with_units == 0]
print(f"\nMice without any units: {nou.subject.tolist()} ({int(nou.sessions.sum())} files)")
print(f"Units per file with units: median {np.median([len(s['n_spikes']) for s in units_files]):.0f}, "
      f"range {min(len(s['n_spikes']) for s in units_files)}-{max(len(s['n_spikes']) for s in units_files)}")

**Result: not reproduced.** The dandiset holds 8,201 units, fewer than the paper's 9,471
(tolerance 8,524 to 10,418). Six of the 17 mice (39 files) have no units at all. The 5 Hz rule
classes 87.8% of units as putative pyramidal cells, against the paper's 78.3% pyramidal by waveform
and autocorrelogram.

### maze-cohort

> Nine mice (E13.5 n = 2, E14.5 n = 2, E15.5 n = 2, E16.5 n = 3) ran a spatial alternation task on a
> familiar figure-eight maze, and a cohort of three mice also explored novel environments.

In [ ]:
maze_mice = subjects[subjects.with_position > 0]
novel_mice = sorted({s["subject"] for s in sessions if s["has_position"] and s["trials"].recordings.nunique() > 1})
print("Mice with figure-eight trials:", len(maze_mice), maze_mice.birthdate.value_counts().sort_index().to_dict())
print("Mice with a second maze recording (novel):", novel_mice)
code_nine = maze_mice[~maze_mice.subject.isin(["e15_13f1", "e16_3m2"])]
print("(a) disjoint cohorts would need 12 maze mice; found", len(maze_mice))
print("(b) maze mice without e15_13f1 and e16_3m2:", len(code_nine), code_nine.birthdate.value_counts().sort_index().to_dict(),
      "; novel mice:", len(novel_mice))
print("Paper: nine mice, E13.5 2, E14.5 2, E15.5 2, E16.5 3; novel cohort 3.")

**Result: reproduced.** Under reading (b), the code's, the maze mice without e15_13f1 and e16_3m2
are nine, at E13.5 2, E14.5 2, E15.5 2 and E16.5 3, and three mice (e13_26m1, e15_13f1, e16_3m2) have
novel recordings. Reading (a) fails: 11 mice have maze data, not 12, because e13_26m1 belongs to both
cohorts, which the paper calls separate.

### maze-task

> On the figure-eight maze each trial was 205 cm long, position was tracked at 30 Hz, and animals
> ran at least ten trials along each arm in every session.

In [ ]:
lens, dts, arms = [], [], []
for s in sessions:
    if not s["has_position"]:
        continue
    fam = familiar(s)
    p = fam["pos"]
    lens += p.groupby("trial").lin.max().tolist()
    dts.append(np.median(np.diff(p.t)))
    t = fam["trials"]
    arms.append({"session": s["session_id"], "subject": s["subject"], "trials": len(s["trials"][s["trials"].recordings == fam["recording"]]),
                 "visited_left": int((t.visited_arm == "left").sum()), "visited_right": int((t.visited_arm == "right").sum()),
                 "expected_left": int((t.expected_arm == "left").sum()), "expected_right": int((t.expected_arm == "right").sum())})
arms = pd.DataFrame(arms)
print(f"Largest linearized position in a familiar trial: median {np.median(lens):.1f} cm (range {np.min(lens):.1f}-{np.max(lens):.1f}); paper 205 cm")
print(f"Tracking rate: median {1 / np.median(dts):.2f} Hz over {len(dts)} familiar recordings (range {1 / np.max(dts):.2f}-{1 / np.min(dts):.2f}); paper 30 Hz")
arms["min_visited"] = arms[["visited_left", "visited_right"]].min(axis=1)
arms["min_expected"] = arms[["expected_left", "expected_right"]].min(axis=1)
print(f"\nSessions with at least 10 trials on each arm: visited {int((arms.min_visited >= 10).sum())} of {len(arms)}; "
      f"expected {int((arms.min_expected >= 10).sum())} of {len(arms)}")
print(f"Sessions with at least 20 trials (rows of the trials table) on the familiar maze: {int((arms.trials >= 20).sum())} of {len(arms)} (fewest {arms.trials.min()})")
print("\nSessions with fewer than 10 on an arm under either reading:")
print(arms[(arms.min_visited < 10) | (arms.min_expected < 10)].to_string(index=False))

**Result: not reproduced.** Trials span 204.1 cm (median of the largest linearized position;
paper 205 cm) and position is sampled at 30.00 Hz, but only 38 of 65 familiar sessions have ten or
more trials on each visited arm (22 of 65 by expected arm), and 6 sessions have fewer than 20 trials
(the fewest has 12).

### novel-explorations

> The novel-environment cohort of three mice made 16 novel environment explorations in six distinct
> environments (two linear tracks, three open fields and one figure-eight maze).

In [ ]:
nov = recs[~recs.familiar]
print("Novel recordings with trials and position:")
print(nov[["session", "recording", "kind", "trials", "lin_max_in_trials", "x_range", "y_range", "dt_ms"]].to_string(index=False))
print(f"\n{len(nov)} novel explorations in {nov.session.str[:8].nunique()} mice; paper 16 in 3 mice (tolerance 14.4-17.6)")
print("Kinds (track length):", nov.groupby(["kind", "lin_max_in_trials"]).size().to_dict())
print("\nEpochs of the novel-cohort sessions (minutes, position samples inside):")
unpos = []
for s in sessions:
    if s["subject"] in novel_mice and "epochs" in s:
        e = s["epochs"].copy()
        e["minutes"] = ((e.stop - e.start) / 60).round(1)
        e["position_samples"] = [int(((s["pos"].t >= a) & (s["pos"].t < b)).sum()) for a, b in zip(e.start, e.stop)]
        print(s["session_id"], e[["minutes", "position_samples"]].values.tolist())
        e["last"] = np.arange(len(e)) == len(e) - 1
        unpos.append(e[(e.position_samples == 0) & e.minutes.between(10, 50) & ~e["last"]].assign(session=s["session_id"]))
unpos = pd.concat(unpos)
print(f"\nEpochs of 10 to 50 min without position that are not a session's last epoch: {len(unpos)}, in sessions {sorted(unpos.session.unique())}")
print("Behavior series in the novel-cohort files:", sorted({str(s["processing"].get("behavior")) for s in sessions if s["subject"] in novel_mice}))
print("Acquisition contents in the novel-cohort files:", sorted({str(s["groups"].get("acquisition")) for s in sessions if s["subject"] in novel_mice}))
print(f"Not counted (no position, so outside the criterion), novel recordings with position plus these epochs: {len(nov)} + {len(unpos)} = {len(nov) + len(unpos)}")

**Result: partially reproduced.** The three mice are there, but only 9 novel explorations have
trials and position (3 T-maze runs of about 164 cm and 6 linear-track runs of 100 or 110 cm),
against 16. The novel-cohort sessions also hold 7 epochs of 10 to 50 min without position that are
not a session's last epoch; the files hold no other position or video series for them, so they are
not counted. They would bring the count to 16, but they were picked after the data were seen, and
nothing in the files says they are explorations. The kinds with position (a T-maze and two track
lengths) match the paper's figure-eight and two linear tracks; no open field has position.

## Claims about all pyramidal neurons

Putative pyramidal cells are units whose session mean rate is at most 5 Hz (the files hold no cell
type). Rate maps follow the Methods (1 cm bins, speed above 1.5 cm/s, 9 cm FWHM smoothing, left and
right trials apart).

### fig5e-pv-decorrelation

> Population firing-rate vectors at the same linearized position decorrelate between left and right
> trials once the animal leaves the common stem of the figure-eight maze (median correlations stem
> 0.93, turn 1 0.412, arm 0.372, turn 2 0.394).

In [ ]:
def pv_correlations(s, max_rate=h.PYR_MAX_RATE_HZ, min_speed=h.MIN_SPEED_CM_S):
    fam = familiar(s)
    keep = h.mean_rates(s) <= max_rate
    sp = [x for x, k in zip(s["maze_spikes"], keep) if k]
    m = h.rate_maps(sp, fam, min_speed=min_speed)["maps"]
    cc = np.full(h.FIG8_BINS, np.nan)
    for b in range(h.FIG8_BINS):
        l, r = m[:, 0, b], m[:, 1, b]
        ok = np.isfinite(l) & np.isfinite(r)
        if ok.sum() > 2 and l[ok].std() > 0 and r[ok].std() > 0:
            cc[b] = np.corrcoef(l[ok], r[ok])[0, 1]
    return cc, int(keep.sum())

pv_sessions = [s for s in units_files if s["has_position"] and s["subject"] not in ("e15_13f1", "e16_3m2")]
pv, ncells = {}, {}
for s in pv_sessions:
    pv[s["session_id"]], ncells[s["session_id"]] = pv_correlations(s)
pvm = np.array(list(pv.values()))

def by_segment(mat):
    return {k: mat[:, a - 1:b].ravel() for k, (a, b) in h.FIG8_SEGMENTS.items()}

seg = by_segment(pvm)
print(f"{len(pv)} sessions of {len({s['subject'] for s in pv_sessions})} mice; putative pyramidal cells per session: median {np.median(list(ncells.values())):.0f}, range {min(ncells.values())}-{max(ncells.values())}")
paper = {"stem": 0.93, "turn1": 0.412, "arm": 0.372, "turn2": 0.3944}
for k, v in seg.items():
    print(f"{k:6s} bins {len(v):5d}, without a value {int(np.isnan(v).sum()):3d}, median {np.nanmedian(v):.3f} "
          f"(paper {paper[k]}, tolerance {paper[k] * 0.9:.3f}-{paper[k] * 1.1:.3f}), range {np.nanmin(v):.2f} to {np.nanmax(v):.2f}")
kw = stats.kruskal(*[v[~np.isnan(v)] for v in seg.values()])
print(f"Kruskal-Wallis across segments: H = {kw.statistic:.1f}, p = {kw.pvalue:.2g} (paper H = 7278.3)")

In [ ]:
sub_of = {s["session_id"]: s["subject"] for s in pv_sessions}
per_mouse = pd.DataFrame({k: pd.Series({m: np.nanmedian(pvm[[sub_of[x] == m for x in pv], a - 1:b])
                                         for m in sorted(set(sub_of.values()))}) for k, (a, b) in h.FIG8_SEGMENTS.items()})
print("Median PV correlation by mouse:")
print(per_mouse.round(3).to_string())
thr_paper = float(np.quantile(rates, 7411 / 9471))
print(f"\nRate threshold that classes the paper's pyramidal fraction ({7411 / 9471:.3f}) of all units as pyramidal: {thr_paper:.2f} Hz")
print("Variants (medians stem, turn1, arm, turn2):")
for label, kw_ in [("all units", dict(max_rate=np.inf)), ("rate at most 10 Hz", dict(max_rate=10)),
                   ("rate at most 2 Hz", dict(max_rate=2)), (f"rate at most {thr_paper:.2f} Hz", dict(max_rate=thr_paper)), ("no speed threshold", dict(min_speed=-1))]:
    mat = np.array([pv_correlations(s, **kw_)[0] for s in pv_sessions])
    sg = by_segment(mat)
    print(f"  {label:20s}", {k: round(float(np.nanmedian(v)), 3) for k, v in sg.items()})
fig, ax = plt.subplots(figsize=(7, 3))
x = np.arange(1, h.FIG8_BINS + 1)
mu, sd = np.nanmean(pvm, 0), np.nanstd(pvm, 0)
ax.fill_between(x, mu - sd, mu + sd, color="0.85")
ax.plot(x, mu, "k")
for a, b in h.FIG8_SEGMENTS.values():
    ax.axvline(b + 0.5, color="0.5", lw=0.5)
ax.set_xlabel("linearized position (cm)"); ax.set_ylabel("PV correlation, left vs right")
ax.set_ylim(-0.2, 1.05)
print(f"Plotted: mean over sessions ({np.nanmin(mu):.2f} to {np.nanmax(mu):.2f}) with one s.d.; "
      f"{int(((mu - sd) < -0.2).sum())} bins of the lower band fall below the axis")
plt.show()

**Result: partially reproduced.** Over 57 sessions of nine mice the population vectors
decorrelate after the stem (Kruskal-Wallis H = 7632.7, p = 0), and the stem median (0.937) matches the
paper's 0.93, but turn 1 (0.547), arm (0.492) and turn 2 (0.465) stay more correlated than reported
(0.412, 0.372, 0.394). The values depend on which units count as pyramidal: with a 2 Hz cut the three
medians are 0.387, 0.288 and 0.306, and with all units 0.948, 0.94 and 0.921. The stem is the most
correlated segment in every mouse (stem 0.849 to 0.952, other segments 0.296 to 0.738). With the
threshold (2.59 Hz) that classes the paper's fraction of units as pyramidal, the four medians are
0.928, 0.436, 0.343 and 0.357, all within tolerance; the criterion fixed the 5 Hz rule, so this
variant does not change the outcome.

### Familiar and novel environments (Figure 6c, d)

The sessions are those of the three mice of the novel-environment cohort (12 sessions, as in the
paper's `get_corrBeh_famnov.m`); 9 of them hold a novel maze recording. The "theta-related" rate is
approximated by the rate while running at 5 cm/s or faster inside maze trials. Running stretches end
at the end of a trial, so the time between trials is not counted; the next cell checks this on
synthetic data and shows how often the animal was running across a trial boundary.

In [ ]:
FAMNOV = ("e13_26m1", "e15_13f1", "e16_3m2")

def famnov_measures(max_rate=h.PYR_MAX_RATE_HZ, run_speed=h.RUN_SPEED_CM_S, include_stem=False):
    cells, pairs, infos = [], [], []
    for s in sessions:
        if s["subject"] not in FAMNOV:
            continue
        keep = h.mean_rates(s) <= max_rate
        sp = [x for x, k in zip(s["maze_spikes"], keep) if k]
        rates = {}
        for r in h.recordings(s):
            m = h.rate_maps(sp, r)
            bins = np.arange(1, r["n_bins"] + 1) if include_stem else h.correlation_bins(r)
            c = h.pairwise_map_correlations(h.concatenated(m["maps"], bins))
            pairs.append(pd.DataFrame({"session": s["session_id"], "subject": s["subject"], "familiar": r["familiar"],
                                       "kind": r["kind"], "r": c}))
            flat = m["maps"].reshape(len(sp), -1)
            infos.append(pd.DataFrame({"session": s["session_id"], "subject": s["subject"], "familiar": r["familiar"],
                                       "kind": r["kind"], "bits": h.skaggs_information(flat, m["occupancy"].ravel()),
                                       "info_unweighted": h.uniform_information(flat)}))
            rates["fam" if r["familiar"] else "nov"] = h.running_rate(sp, r, min_speed=run_speed)
        cells.append(pd.DataFrame({"session": s["session_id"], "subject": s["subject"], "fam": rates["fam"],
                                   "nov": rates.get("nov", np.full(len(sp), np.nan))}))
    return pd.concat(cells, ignore_index=True), pd.concat(pairs, ignore_index=True), pd.concat(infos, ignore_index=True)

cells6, pairs6, info6 = famnov_measures()
print(f"{cells6.session.nunique()} sessions, {len(cells6)} putative pyramidal cells; "
      f"{cells6.nov.notna().sum()} in the {cells6[cells6.nov.notna()].session.nunique()} sessions with a novel maze")

In [ ]:
# The running rate on synthetic data: two 10 s trials 10 s apart, running throughout, a unit firing at 1 Hz
# also between the trials. The time between trials must not count, so the rate is 1 Hz.
t_syn = np.r_[np.arange(0, 10, 1 / 30), np.arange(20, 30, 1 / 30)]
pos_syn = pd.DataFrame({"t": t_syn, "speed": 10.0, "trial": np.repeat([0, 1], 300)})
r_syn = h.running_rate([np.arange(0.5, 30, 1.0)], {"pos": pos_syn})[0]
print(f"Synthetic running rate: {r_syn:.3f} Hz (true 1 Hz)")
assert abs(r_syn - 1) < 1e-9

# Gaps between consecutive trials, and running at the edges of trials, in the sessions of the three mice
gap_rows = []
for s in sessions:
    if s["subject"] not in FAMNOV:
        continue
    for r in h.recordings(s):
        tr = r["trials"].sort_values("start_time")
        p = r["pos"]
        edge = p.groupby("trial").speed.agg(["first", "last"]).reindex(tr.index)
        gap_rows.append({"familiar": r["familiar"], "gap_s": (tr.start_time.values[1:] - tr.stop_time.values[:-1]),
                         "run_across": (edge["last"].values[:-1] >= h.RUN_SPEED_CM_S) & (edge["first"].values[1:] >= h.RUN_SPEED_CM_S)})
for fam in (True, False):
    g = np.concatenate([x["gap_s"] for x in gap_rows if x["familiar"] == fam])
    ra = np.concatenate([x["run_across"] for x in gap_rows if x["familiar"] == fam])
    print(f"{'Familiar' if fam else 'Novel'}: gaps between consecutive trials median {np.median(g):.2f} s (range {g.min():.2f} to {g.max():.1f}); "
          f"pairs of trials running at the end of one and the start of the next: {ra.sum()} of {len(ra)}")


### fig6c-theta-rate-preserved

> Individual pyramidal neurons preserve their theta-related firing rates between the familiar and
> novel environments (Spearman rho = 0.7378, n = 2,544 neurons).

In [ ]:
both = cells6.dropna(subset=["fam", "nov"])
print(f"Cells with both rates: {len(both)}; dropped: {cells6.nov.isna().sum()} in sessions without a novel maze, "
      f"{(cells6.fam.isna() & cells6.nov.notna()).sum()} without a familiar rate")
r = stats.spearmanr(both.fam, both.nov)
print(f"Spearman rho = {r.statistic:.3f}, p = {r.pvalue:.2g}, n = {len(both)} (paper 0.7378, n = 2,544; criterion rho >= 0.664)")
print("Cells silent in one environment (rate 0): familiar", int((both.fam == 0).sum()), ", novel", int((both.nov == 0).sum()))
print("\nBy mouse:")
for m, g in both.groupby("subject"):
    rr_ = stats.spearmanr(g.fam, g.nov)
    print(f"  {m}: rho = {rr_.statistic:.3f}, p = {rr_.pvalue:.2g}, n = {len(g)}")
print("By session:", {k: round(stats.spearmanr(g.fam, g.nov).statistic, 3) for k, g in both.groupby("session")})
print("\nVariants:")
for label, kw_ in [("all units", dict(max_rate=np.inf)), ("rate at most 10 Hz", dict(max_rate=10)), ("running at 1.5 cm/s or faster", dict(run_speed=1.5))]:
    cv = famnov_measures(**kw_)[0].dropna(subset=["fam", "nov"])
    rv = stats.spearmanr(cv.fam, cv.nov)
    print(f"  {label:30s} rho = {rv.statistic:.3f}, n = {len(cv)}")
fig, ax = plt.subplots(figsize=(3.5, 3.5))
lo = 0.01
ax.loglog(np.clip(both.fam, lo, None), np.clip(both.nov, lo, None), ".", ms=2, color="0.3")
ax.set_xlabel("familiar rate (Hz)"); ax.set_ylabel("novel rate (Hz)")
print(f"Plotted rates range {both[['fam', 'nov']].min().min():.3f} to {both[['fam', 'nov']].max().max():.2f} Hz; "
      f"{int(((both.fam < lo) | (both.nov < lo)).sum())} cells below {lo} Hz drawn at {lo}")
plt.show()

**Result: reproduced.** The running rates of 2,038 putative pyramidal cells on the familiar and
novel mazes correlate with Spearman rho = 0.726 (paper 0.7378), and rho is 0.711 to 0.745 in each
mouse; the variants give 0.754 to 0.802. The rate here is a running rate, not the paper's rate in
theta cycles.

### fig6c-novel-ratemap-corr-higher

> Pairwise spatial ratemap correlations between pyramidal neurons are higher in novel environments
> (median 0.048) than in the familiar environment (median -0.045).

In [ ]:
fc = pairs6[pairs6.familiar].r.dropna(); nc = pairs6[~pairs6.familiar].r.dropna()
dropped = pairs6.r.isna().sum()
t = stats.ranksums(nc, fc)
print(f"Familiar: median {fc.median():.4f}, n = {len(fc)} pairs (paper -0.045, n = 31,067; tolerance -0.075 to -0.015)")
print(f"Novel:    median {nc.median():.4f}, n = {len(nc)} pairs (paper 0.048, n = 41,991; tolerance 0.018 to 0.078)")
print(f"Rank-sum (novel against familiar): z = {t.statistic:.2f}, p = {t.pvalue:.2g}; pairs without a value (a silent cell): {dropped}")
print("\nBy mouse and maze (median, pairs):")
print(pairs6.groupby(["subject", "familiar", "kind"]).r.agg(["median", "count"]).round(4).to_string())
print("\nBy session, familiar against novel median in sessions with both:")
ps = pairs6.groupby(["session", "familiar"]).r.median().unstack()
print(ps.dropna().round(4).to_string())
print("Sessions where the novel median is higher:", int((ps.dropna()[False] > ps.dropna()[True]).sum()), "of", len(ps.dropna()))
n_hi, n_ses = int((ps.dropna()[False] > ps.dropna()[True]).sum()), len(ps.dropna())
print(f"Variant, sign test over sessions (novel median higher in {n_hi} of {n_ses}): p = {stats.binomtest(n_hi, n_ses).pvalue:.2g}")
print("Variant, rank-sum p within each session (novel against familiar):",
      {k: float(f"{stats.ranksums(g[~g.familiar].r.dropna(), g[g.familiar].r.dropna()).pvalue:.2g}") for k, g in pairs6[pairs6.session.isin(ps.dropna().index)].groupby("session")})
print("Pairs per session (familiar, novel):", pairs6.dropna(subset=["r"]).groupby(["session", "familiar"]).size().unstack().fillna(0).astype(int).values.tolist())
fam_same = pairs6[pairs6.familiar & pairs6.session.isin(ps.dropna().index)].r.dropna()
print(f"\nVariant, familiar pairs only from sessions with a novel maze: median {fam_same.median():.4f}, rank-sum p = {stats.ranksums(nc, fam_same).pvalue:.2g}")
for label, kw_ in [("all units", dict(max_rate=np.inf)), ("rate at most 10 Hz", dict(max_rate=10)), ("stem included", dict(include_stem=True))]:
    pv_ = famnov_measures(**kw_)[1]
    f_, n_ = pv_[pv_.familiar].r.dropna(), pv_[~pv_.familiar].r.dropna()
    print(f"Variant, {label:20s} familiar {f_.median():.4f}, novel {n_.median():.4f}, rank-sum p = {stats.ranksums(n_, f_).pvalue:.2g}")

**Result: not reproduced.** Pooled over pairs, the median ratemap correlation is -0.0364 in
novel mazes and -0.0530 on the familiar maze, the rank-sum test finds no difference (p = 0.67), and
the novel median is outside its tolerance (0.018 to 0.078). Within each of the 9 sessions with a
novel maze the novel median is the higher one (sign test p = 0.0039; the rank-sum test within a
session gives p < 0.05 in 7 of 9), so the direction holds session by session, but the pooled test the
criterion names and the reported size do not. The pooled test is weighted by the sessions with most
pairs. The dandiset lacks the open-field explorations, and the
paper's pairs each include a light-responsive neuron; either may account for the difference.

### fig6d-info-lower-novel

> Spatial information is lower in novel environments than in the familiar one (median 0.39 against
> 0.88 bits/s).

In [ ]:
fi = info6[info6.familiar].bits.dropna(); ni = info6[~info6.familiar].bits.dropna()
t = stats.ranksums(ni, fi)
print(f"Familiar: median {fi.median():.3f} bits/s, n = {len(fi)} (paper 0.88, n = 2,544; tolerance 0.792-0.968)")
print(f"Novel:    median {ni.median():.3f} bits/s, n = {len(ni)} (paper 0.39, n = 3,443; tolerance 0.351-0.429)")
print(f"Rank-sum: z = {t.statistic:.2f}, p = {t.pvalue:.2g}; cells without a value (silent on the maze): {int(info6.bits.isna().sum())}")
print(f"Ranges: familiar {fi.min():.3f} to {fi.max():.2f}, novel {ni.min():.3f} to {ni.max():.2f}")
print("\nBy mouse and maze (median bits/s, cells):")
print(info6.groupby(["subject", "familiar", "kind"]).bits.agg(["median", "count"]).round(3).to_string())
fu, nu = info6[info6.familiar].info_unweighted.dropna(), info6[~info6.familiar].info_unweighted.dropna()
print(f"\nVariant, the code's unweighted information: familiar {fu.median():.3f}, novel {nu.median():.3f}, rank-sum p = {stats.ranksums(nu, fu).pvalue:.2g}")
for label, kw_ in [("all units", dict(max_rate=np.inf)), ("rate at most 10 Hz", dict(max_rate=10))]:
    iv = famnov_measures(**kw_)[2]
    print(f"Variant, {label:20s} familiar {iv[iv.familiar].bits.median():.3f}, novel {iv[~iv.familiar].bits.median():.3f}")

**Result: partially reproduced.** Spatial information is lower on novel mazes (median 0.270
bits/s) than on the familiar maze (0.631 bits/s), rank-sum p = 1.5e-59, as the paper reports, but
both medians are below the paper's 0.39 and 0.88. The code's unweighted information gives 0.285 and
0.646. Novel information is lowest on the linear tracks (0.083 to 0.337) and, for one mouse's T-maze,
close to its familiar value (0.608 against 0.617).

### fig6d-rate-lower-novel

> Theta-related firing rates are slightly lower in novel environments than in the familiar one
> (median 0.69 against 0.77 Hz).

In [ ]:
fr_, nr_ = cells6.fam.dropna(), cells6.nov.dropna()
t = stats.ranksums(nr_, fr_)
print(f"Familiar: median {fr_.median():.3f} Hz, n = {len(fr_)} (paper 0.77, n = 2,544; tolerance 0.693-0.847)")
print(f"Novel:    median {nr_.median():.3f} Hz, n = {len(nr_)} (paper 0.69; tolerance 0.621-0.759)")
print(f"Rank-sum: z = {t.statistic:.2f}, p = {t.pvalue:.2g} (paper p = 3.3e-3)")
w = stats.wilcoxon(both.nov, both.fam)
print(f"Variant, paired signed-rank over {len(both)} cells with both: median ratio novel/familiar "
      f"{np.median(both.nov[both.fam > 0] / both.fam[both.fam > 0]):.3f}, p = {w.pvalue:.2g}")
fam_same = cells6[cells6.nov.notna()].fam
print(f"Variant, familiar only in sessions with a novel maze: median {fam_same.median():.3f} Hz, rank-sum p = {stats.ranksums(nr_, fam_same).pvalue:.2g}")
print("\nBy mouse (median Hz):")
print(cells6.groupby("subject")[["fam", "nov"]].median().round(3).to_string())
for label, kw_ in [("all units", dict(max_rate=np.inf)), ("rate at most 10 Hz", dict(max_rate=10)), ("running at 1.5 cm/s or faster", dict(run_speed=1.5))]:
    cv = famnov_measures(**kw_)[0]
    print(f"Variant, {label:30s} familiar {cv.fam.median():.3f}, novel {cv.nov.median():.3f}, rank-sum p = {stats.ranksums(cv.nov.dropna(), cv.fam.dropna()).pvalue:.2g}")

**Result: reproduced.** Running rates are lower on novel mazes (median 0.714 Hz) than on the
familiar maze (0.794 Hz), rank-sum p = 0.041, and both medians are within tolerance of the paper's
0.69 and 0.77 Hz. The difference is weak: the paired test over 2,038 cells gives a median
novel-to-familiar ratio of 0.942 (p = 0.19), the variants give p = 0.075 to 0.11, and in mouse
e15_13f1 the novel median is slightly higher (0.734 against 0.729 Hz).

## Blocked claims

The data check shows what the files lack. Every claim that compares same-birthdate (SBD) with
different-birthdate (DBD) neurons, or light-responsive neurons across birthdates, needs to know
which units responded to light. The files hold no such label, no optogenetic pulse times from which
to derive one (neither the processed nor the raw files have a stimulation channel), and no
waveforms. The paper's labels come from CellExplorer's `cell_metrics.optoTag` and
`putativeCellType` fields and `pulses.events.mat` files on the lab's server, which are not public
at a pinned identifier. Claims fig5d, fig7b and fig7ce also need unit positions on the probe or cell
types, which the files lack. fig1c needs histology, which is not in the dandiset. fig7fg needs
putative interneurons and their monosynaptic inputs. The cell below applies the rate rule (units
above 5 Hz as interneurons) and the Methods' connection test to one session: it finds connections in
a far larger fraction of pairs than the paper does over its dataset, so this procedure is not the
paper's, which comes from the lab's CellExplorer files (`mono_res`), not public. The units of 31 of
the paper's 96 sessions are also not in the dandiset.

| Claim | Reason |
|---|---|
| ed2-light-responsive-count, fig1e, fig1f, fig1g, fig1h | no light-responsiveness labels or pulse times |
| fig2b, fig2c, fig2d, fig2f, fig2g | no labels; SBD and DBD pairs cannot be formed |
| fig3c, fig3e | no labels; assemblies cannot be grouped by birthdate |
| fig4d-model-fit | depends on fig3e |
| fig5b, fig5c, fig5f, fig6e, fig6f | no labels |
| fig5d | no labels and no unit positions |
| fig7b, fig7ce | no labels and no cell types |
| fig1c-radial-depth | no histology |
| fig7fg-convergence-cofiring | connection detection differs from the paper's (methods underspecified) |

In [ ]:
# What a test of Figure 7f would take: interneurons by the rate rule, and connections on one session.
n_int = pd.DataFrame([{"session": s["session_id"], "units": len(s["n_spikes"]),
                       "above_5Hz": int((h.mean_rates(s) > h.PYR_MAX_RATE_HZ).sum())} for s in units_files])
print(f"Units above {h.PYR_MAX_RATE_HZ} Hz per file with units: median {n_int.above_5Hz.median():.0f}, range {n_int.above_5Hz.min()}-{n_int.above_5Hz.max()}, "
      f"total {n_int.above_5Hz.sum()} (paper 2,060 interneurons)")
print(f"Files with units: {len(units_files)} of the paper's 96 sessions of Figure 7f")
ex = max(units_files, key=lambda s: len(s["n_spikes"]))
trains = h.spike_trains(proc[proc.path == ex["path"]].asset.iloc[0])
mr = h.mean_rates(ex)
pyr_i, int_i = np.where(mr <= h.PYR_MAX_RATE_HZ)[0], np.where(mr > h.PYR_MAX_RATE_HZ)[0]
t0 = time.time()
targets = {p: {i for i in int_i if h.is_monosynaptic(h.ccg_counts(trains[p], trains[i]))} for p in pyr_i}
n_conn = sum(len(v) for v in targets.values())
conv = []
for a in range(len(pyr_i)):
    for b in range(a + 1, len(pyr_i)):
        u = targets[pyr_i[a]] | targets[pyr_i[b]]
        conv.append(len(targets[pyr_i[a]] & targets[pyr_i[b]]) / len(u) if u else np.nan)
conv = np.array(conv)
print(f"\nExample session {ex['session_id']}: {len(pyr_i)} units at or below and {len(int_i)} above {h.PYR_MAX_RATE_HZ} Hz; "
      f"{n_conn} connections detected of {len(pyr_i) * len(int_i)} possible ({n_conn / (len(pyr_i) * len(int_i)):.3f}; paper 31,932 of 184,230 = {31932 / 184230:.3f}) "
      f"in {time.time() - t0:.0f} s")
print(f"Pyramidal pairs: {len(conv)}; nonzero convergence {np.mean(conv > 0):.3f}, zero {np.mean(conv == 0):.3f}, no target in either {np.mean(np.isnan(conv)):.3f} "
      f"(paper over 96 sessions 0.6527, 0.3416)")
print(f"Interneurons with at least one detected input: {len(set().union(*targets.values()))} of {len(int_i)}")


In session e15_13f1_220117, the 343 units at or below 5 Hz and the 33 above give 5,073 detected
connections in 11,319 possible pairs (0.448, against 0.173 in the paper's dataset), and 96.7% of
pyramidal pairs share a target (paper 65.27%). Over all files the rate rule finds 1,001 units above
5 Hz, against the paper's 2,060 interneurons. The procedure here does not give the paper's rate of
connections, so the claim stays blocked.

In [ ]:
print(f"Run time: {(time.time() - started) / 60:.1f} min")

## Summary

| Claim | Outcome |
|---|---|
| cohort-birthdates | reproduced |
| sessions-durations | partially reproduced |
| spwr-duration | reproduced |
| unit-count | not reproduced |
| maze-cohort | reproduced |
| maze-task | not reproduced |
| novel-explorations | partially reproduced |
| fig5e-pv-decorrelation | partially reproduced |
| fig6c-theta-rate-preserved | reproduced |
| fig6c-novel-ratemap-corr-higher | not reproduced |
| fig6d-info-lower-novel | partially reproduced |
| fig6d-rate-lower-novel | reproduced |
| ed2, fig1e, fig1f, fig1g, fig1h, fig2b, fig2c, fig2d, fig2f, fig2g, fig3c, fig3e, fig4d, fig5b, fig5c, fig5d, fig5f, fig6e, fig6f, fig7b, fig7ce, fig1c | blocked: data missing |
| fig7fg-convergence-cofiring | blocked: methods underspecified |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

**How the analysis differs from the paper's.** Putative pyramidal cells are units with a session
mean rate of at most 5 Hz, as the files hold no cell types or waveforms; the Figure 5e values move a
long way with this choice, and a 2.59 Hz rule, which classes the paper's fraction as pyramidal, puts
all four within tolerance. The "theta-related" rate is the rate while running at 5 cm/s or faster,
not the rate in LFP theta cycles; its Figure 6d difference is weak (p = 0.041, paired p = 0.19).
Speed comes from smoothed linearized position, not a Kalman filter.
The Figure 6c pair correlations use all pairs of putative pyramidal cells, where the paper's pairs
each include a light-responsive neuron, and the novel mazes are only those with position (no open
fields). The familiar maze is taken as the first maze recording of each session.

**What was found about the dandiset.**

- No unit is labelled light-responsive, no cell type is given, no waveforms are stored and no
  optogenetic pulse times are stored, in the processed or the raw files. The paper's main findings,
  which compare same- and different-birthdate neurons, cannot be tested.
- Six of the 17 mice (39 files) have sleep states and ripples but no units and no position; the
  dandiset holds 8,201 units where the paper reports 9,471.
- No open field has position: only 9 of the 16 novel explorations have trials and position, and
  the novel-cohort sessions hold 7 further epochs without position that may be the others. Several novel recordings give x and y between 0 and 1 though stated in cm.
- `is_familiar_maze` is wrong in sessions with a single maze recording (True only for the first two
  trials). The sex field differs between files
  of e13_1f1 (Female 3, Male 2) and e13_1m1 (Male 9, Female 3). Two files of e13_1f1 have no scored sleep.
- Twelve of 13 raw files give a session start of 2011-08-18. The processed file of e16_3m1_210201
  holds a copy of the 1250 Hz LFP labelled 30,000 Hz with a wrong conversion factor.
- Units have no link to electrodes, and their `group_id` values do not match the electrode group
  names in 64 of 65 files. The birthdate of each mouse is stored only in its subject id.
- The 96 files of the 15 homecage mice match the paper's per-birthdate session counts and duration
  bounds exactly; the Methods' count of 92 sessions disagrees with the paper's own 96.